# MAC 237 Graded Lab 3, Session 12: storing a password so that stealing the file is not enough

**Wednesday, October 28, 2026. This session runs in Google Colab, in a browser.**

B124 is not available to us on Wednesday evenings, so this session is a notebook you run on
any computer with a browser: a classroom machine or your own laptop. It uses only the Python
3 standard library, needs no administrative rights, opens no socket, and writes only inside
its own folder.

**Nothing in this notebook contacts any host outside it.**

The failure this session is about sits at number seven in the OWASP Top Ten as identification
and authentication failures, and it is almost always the same failure: a password database
that is fast to attack once it is copied. Three properties fix it, and this notebook measures
each one.

## Section 1. Your personalization token

Put your own student ID in `sid` and run the cell. The token must be visible in the first and
the last screenshot of this session.

In [ ]:
import hashlib, datetime

sid = '24621670'          # put your own student ID here, in quotes
stamp = datetime.datetime.now().strftime('%Y%m%d-%H%M')
TOKEN = hashlib.sha256((sid + '-' + stamp).encode()).hexdigest()[:16]
print('MAC237 Lab 3 token:', TOKEN)

MAC237 Lab 3 token: 14ed3751f1acdaa5


## Section 6. Salt, work factor and a constant-time comparison

The password below is built from your token, so your numbers are yours. The cell stores it two
ways: an unsalted SHA-256 digest, which is what a database that was never designed for this
holds, and a salted scrypt digest with a deliberate work factor. Then it measures what one guess
costs under scrypt on this machine and converts both into guesses per hour.

In [ ]:
import hashlib, hmac, os, time

password = ('correct horse ' + TOKEN).encode()

fast = hashlib.sha256(password).hexdigest()
salt = os.urandom(16)
t0 = time.perf_counter()
slow = hashlib.scrypt(password, salt=salt, n=2**14, r=8, p=1, dklen=32)
cost = time.perf_counter() - t0

def verify(candidate):
    got = hashlib.scrypt(candidate.encode(), salt=salt, n=2**14, r=8, p=1, dklen=32)
    return hmac.compare_digest(got, slow)

ok, bad = verify('correct horse ' + TOKEN), verify('correct horse wrong')
print('token           :', TOKEN)
print('unsalted sha256 :', fast)
print('salt            :', salt.hex())
print('scrypt digest   :', slow.hex())
print('cost per guess  : %.3f seconds' % cost)
print('verify correct  :', ok)
print('verify wrong    :', bad)
print('one hour of guessing at 1 microsecond each : %.1e' % (3600 / 1e-6))
print('one hour of guessing at %.3f s each        : %.1e' % (cost, 3600 / cost))

assert ok is True and bad is False, 'verify must accept the password and nothing else'
assert cost > 1e-4, 'scrypt should take measurably longer than a plain digest'
print()
print('  CHECK PASS: the salted, slow digest verifies correctly and costs a measurable time per guess')

token           : 14ed3751f1acdaa5
unsalted sha256 : fc864488a52e0469b6f89965fab6c5d21a9d7026bc798b4ebe65ec34a6935c6b
salt            : c00aa9998a179c9e0f247baa6091612b
scrypt digest   : 01cde866da7bea0ec96d4412840faedec27e8f2ca685855be3968b7d7170af43
cost per guess  : 0.060 seconds
verify correct  : True
verify wrong    : False
one hour of guessing at 1 microsecond each : 3.6e+09
one hour of guessing at 0.060 s each        : 6.0e+04

  CHECK PASS: the salted, slow digest verifies correctly and costs a measurable time per guess


## What goes in the report

The `CHECK PASS` line must appear above.

1. Name the three properties and what each one costs an attacker, using your own two numbers
   rather than numbers from a slide. The salt: why a per-user random value defeats a precomputed
   table, and why it does not need to be secret. The work factor: your measured guess rate with
   and without it, and the ratio between them. The comparison: why `hmac.compare_digest` is used
   instead of the equality operator, and what an attacker learns from a comparison that returns
   early.
2. Your measured cost per guess is also a cost your own login page pays on every sign-in. Say how
   you would choose the work factor for a real service.

**Screenshot** `lab3_s12_password_store.png`, showing the two guess-rate lines and True followed
by False, with your token and your Google account name.

**Save and sign out.** Save the notebook into `lab3` with File, then Save a copy in GitHub,
under this notebook's name. Choose Runtime, then Disconnect and delete runtime, and on a shared
computer sign out of Google. Capture `lab3_s12_logout.png` while the token is still on
screen, then upload both screenshots to `lab3` on github.com with Add file, then Upload files.